In [ ]:
!pip install uv

In [ ]:
!uv pip install openai pandas arize arize-otel openinference-instrumentation-openai openai-agents

In [ ]:
!git clone https://github.com/rk-yen/customer-support-collab.git

# Secrets (Before Starting) ...
Add the following variables in the Secrets section:
* ARIZE_API_KEY
* ARIZE_SPACE_ID
* OPENAI_API_KEY

In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "workshop_helpers").exists():
    candidates = [path for path in REPO_ROOT.iterdir() if path.is_dir() and (path / "workshop_helpers").exists()]
    if not candidates:
        raise FileNotFoundError("Could not find a repo folder containing workshop_helpers")
    REPO_ROOT = candidates[0]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workshop_helpers.backend import (
    TOOLS,
    default_support_agent_instructions,
    hydrate_backend_from_dataset,
    run_support_agent_async,
    snapshot_backend,
    support_agent_instructions_template,
)
from workshop_helpers.data import DATASET
from workshop_helpers.experiments import (
    build_v2_support_snapshot,
    build_workshop_benchmark,
    format_checklist_rows,
    prepare_experiment_bundle,
    production_readiness_checklist,
    run_experiment,
    summarize_dataset,
)
from workshop_helpers.metrics import compare_scores
from workshop_helpers.scenarios import run_context_agent, run_raw_llm


In [ ]:
from workshop_helpers.setup import setup_clients

env = setup_clients()
client = env["client"]
ARIZE_SPACE_ID = env["arize_space_id"]
ARIZE_API_KEY = env["arize_api_key"]

---
## Step 1 — Raw LLM Call

The simplest possible baseline: a two-sentence system prompt, no customer data.
This is what most teams' v0 looks like — pasting into ChatGPT.

In [ ]:
PROMPT_V1 = "You are a customer support agent. Help the customer."
CUSTOMER_MESSAGE = (
    "I received the wrong colour. I ordered the navy blue tote but got black instead."
)

DEMO_CASE = {
    "scenario_id": "CS_015",
    "category": "returns",
    "benchmark_slice": "tools",
    "expected_behavior": "confirm_backend_action_taken",
    "user_input": CUSTOMER_MESSAGE,
    "source_data": {
        "customer_name": "Jack Morrison",
        "customer_id": "CUST_5540",
        "order_id": "ORD_3318",
        "product_name": "Classic Canvas Tote - Navy Blue",
        "order_date": "2024-03-17",
        "days_since_order": 5,
        "order_total": 32.0,
        "order_status": "delivered",
        "return_policy_days": 30,
        "account_status": "active",
        "notes": "Warehouse pick error confirmed in fulfilment log. Navy Blue variant in stock.",
    },
    "expected_output": "Hi Jack, I'm really sorry about this - that's a fulfilment error on our side and I apologise for the inconvenience. I'll ship the correct Navy Blue Classic Canvas Tote to you today at no extra charge. I'll also include a prepaid return label for the black one, though there's no rush on sending it back. You should receive dispatch details within a few hours.",
}

output_v1 = run_raw_llm(client, CUSTOMER_MESSAGE, PROMPT_V1)

print("V1 (raw LLM):")
print("-" * 50)
print(output_v1)
print("-" * 50)


**What is missing?**  The model has no idea who the customer is, what they ordered,
or what your return policy says. It can only ask for more information — or worse,
make something up.

## Update PROMPT_V1
Use https://platform.openai.com/chat/edit?models=gpt-4o-mini&optimize=true and update `PROMPT_V1`

### Objective
_Improve this baseline customer support system prompt for cases where the model only has the customer's message and no backend access. The response should be empathetic, concise, and honest about uncertainty. It must not invent order details, policy details, dates, or actions taken. When information is missing, it should ask only the single most useful follow-up question. Optimize for better tone and clearer issue handling, while keeping the limitations of a raw prompt-only assistant explicit._

### What to watch for
A stronger prompt should improve `v1` on **prompt-only behavior**: better tone, sharper follow-up questions, and less bluffing.
It should not make `v1` behave like `v2` or `v3`, because it still has no context and no tools.


---
## Step 2 — Add Context

Inject the customer's account data directly into the prompt.
No tools yet — just better information.
This is the **prompted agent** pattern.

In [ ]:
import json

def build_context_message(customer_message: str, source_data: dict) -> str:
    context_block = json.dumps(source_data, indent=2)
    return f"Customer context (internal):\n{context_block}\n\nCustomer message:\n{customer_message}"

SOURCE_DATA = DEMO_CASE["source_data"]
V2_CONTEXT = build_v2_support_snapshot(DEMO_CASE)

PROMPT_V2 = (
    "You are a customer support agent looking at an internal support snapshot. "
    "Use the provided context carefully and be specific. "
    "Do not claim that you already issued a refund, sent a label, escalated a case, or made any backend change. "
    "If the snapshot is enough to explain the likely next step, do that clearly. "
    "If a backend action would still need to happen, describe the next step without pretending it is already done."
)

print("Context passed to the model:")
customer_message_with_context = build_context_message(CUSTOMER_MESSAGE, V2_CONTEXT)
print(customer_message_with_context)
print()

output_v2 = run_context_agent(client, customer_message_with_context, PROMPT_V2)

print("V2 (with context):")
print("-" * 50)
print(output_v2)
print("-" * 50)


### Scoring — three metrics we will reuse throughout

| Metric | Type | What it checks |
|--------|------|----------------|
| `correct_outcome` | AI judge | Did the response reach the right substantive outcome? |
| `workflow_fit` | AI judge | Did it behave like the intended stage: prompt, context, or tools? |
| `tone_quality` | AI judge | Was it empathetic and professional? |


In [ ]:
scorecard = pd.DataFrame(
    compare_scores(
        client,
        {
            "V1 raw LLM": output_v1,
            "V2 + context": output_v2,
        },
        case=DEMO_CASE,
    )
)

display(scorecard[["variant", "correct_outcome", "workflow_fit", "tone", "total"]])


---
## Step 3 — Tool-Calling Agent (OpenAI Agents SDK)

V2 trusts whatever we inject. If `days_since_order` is stale in the database,
the model uses the wrong value. Policy arithmetic ("is this within 30 days?") is
just math — a model should not guess at it.

**Tools** let the model call real functions: it decides *what* to look up,
and *when* to act. That is the difference between a prompted assistant and an agent.

We use the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/),
which handles the tool-calling loop, tracing, and function schema generation
from Python type hints and docstrings.

```
  Customer message
       |
       v
  [ gpt-4o-mini ]  -- get_customer_profile(CUST_6601) -->  ORDER_DB + ACCOUNT_DB
       |                                                          |
       |          <-- profile: orders, subscription, app --------+
       |
       +-- check_return_eligibility(ORD_8814) --> 35 days, outside window by 5
       |
       v
  Declines + offers supervisor escalation
```

In [ ]:
backend_snapshot = snapshot_backend()

print("Simulated systems ready:")
for label, value in backend_snapshot.items():
    print(f"  {label}: {value}")


In [ ]:
print("Tools available to the agent:")
for tool in TOOLS:
    print(f"  - {tool.name}")


In [ ]:
AUTHENTICATED_CUSTOMER_ID = SOURCE_DATA["customer_id"]
PROMPT_V3 = support_agent_instructions_template()

agent_result = await run_support_agent_async(
    customer_message=CUSTOMER_MESSAGE,
    authenticated_customer_id=AUTHENTICATED_CUSTOMER_ID,
    instructions=PROMPT_V3.format(authenticated_customer_id=AUTHENTICATED_CUSTOMER_ID),
)
output_v3 = agent_result.final_output
tool_calls = [
    raw.name
    for item in agent_result.new_items
    for raw in [getattr(item, "raw_item", None)]
    if raw and hasattr(raw, "name")
]

print("PROMPT_V3 template:")
print(PROMPT_V3)
print()
print("V3 (Agents SDK):")
print("-" * 50)
print(output_v3)
print("-" * 50)
print()
print("Tool call chain:")
for name in tool_calls:
    print(f"  -> {name}()")


In [ ]:
scorecard = pd.DataFrame(
    compare_scores(
        client,
        {
            "V1 raw LLM": output_v1,
            "V2 context": output_v2,
            "V3 tools": output_v3,
        },
        case=DEMO_CASE,
    )
)

display(scorecard[["variant", "correct_outcome", "workflow_fit", "tone", "total"]])

print("What tools add over context injection:")
print("  - The agent can verify the order details instead of trusting a static snapshot")
print("  - The agent can execute the replacement workflow instead of describing it")
print("  - The same agent shell can extend across billing, shipping, access, and subscriptions")


---
## Step 4 — Arize Datasets + Experiments

One demo case builds intuition. **Experiments** answer the harder question:
does the improvement hold across a curated workshop benchmark?

We use an **8-case benchmark** balanced across three slices:
1. **Prompt**: the model should ask a sharp follow-up instead of bluffing
2. **Context**: a static support snapshot should answer correctly without claiming actions
3. **Tools**: the agent should verify facts and complete backend actions

The benchmark is intentionally small and opinionated. It is designed to make the capability progression visible in Arize, not to represent every support workflow equally.


In [ ]:
WORKSHOP_BENCHMARK = build_workshop_benchmark(DATASET)

library_summary = summarize_dataset(DATASET)
benchmark_summary = summarize_dataset(WORKSHOP_BENCHMARK)

print(f"Scenario library: {library_summary['scenario_count']} total cases")
print(f"Workshop benchmark: {benchmark_summary['scenario_count']} cases")
print(f"  Categories: {', '.join(benchmark_summary['categories'])}")
print("  Slice counts:")
for slice_name, count in benchmark_summary["slice_counts"].items():
    print(f"    - {slice_name}: {count}")

pd.DataFrame(WORKSHOP_BENCHMARK)[["scenario_id", "category", "benchmark_slice", "expected_behavior"]]


In [ ]:
backend_snapshot = hydrate_backend_from_dataset(DATASET)
print("Backend refreshed from the scenario library:")
for label, value in backend_snapshot.items():
    print(f"  {label}: {value}")


In [ ]:
experiment_bundle = prepare_experiment_bundle(
    client=client,
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=WORKSHOP_BENCHMARK,
    prompt_v1=PROMPT_V1,
    prompt_v2=PROMPT_V2,
    prompt_v3=PROMPT_V3,
)

arize_client = experiment_bundle["client"]
DATASET_ID = experiment_bundle["dataset_id"]

print(f"Dataset name: {experiment_bundle['dataset_name']}")
print(f"Dataset ID:   {DATASET_ID}")
print(f"Rows:         {experiment_bundle['row_count']}")
print(f"Created now:  {experiment_bundle['created']}")


In [ ]:
tasks = experiment_bundle["tasks"]

print("Tasks defined:")
for name in tasks:
    print(f"  - {name}")


In [ ]:
EVALUATORS = experiment_bundle["evaluators"]

print("Evaluators ready:")
for evaluator in EVALUATORS:
    print(f"  - {evaluator.__class__.__name__}")


In [ ]:
print("Running experiment 1/3: v1-raw-llm...")
run_v1 = run_experiment(
    arize_client=arize_client,
    dataset_id=DATASET_ID,
    name_prefix="v1-raw-llm",
    task=tasks["task_v1"],
    evaluators=EVALUATORS,
)
exp_v1 = run_v1["experiment"]
df_v1 = run_v1["results_df"]
print(f"  Done. Experiment ID: {exp_v1.id}")
print(f"  Rows evaluated: {len(df_v1)}")


In [ ]:
print("Running experiment 2/3: v2-context-agent...")
run_v2 = run_experiment(
    arize_client=arize_client,
    dataset_id=DATASET_ID,
    name_prefix="v2-context-agent",
    task=tasks["task_v2"],
    evaluators=EVALUATORS,
)
exp_v2 = run_v2["experiment"]
df_v2 = run_v2["results_df"]
print(f"  Done. Experiment ID: {exp_v2.id}")
print(f"  Rows evaluated: {len(df_v2)}")


In [ ]:
print("Running experiment 3/3: v3-tool-agent (async tasks, takes a few minutes)...")
run_v3 = run_experiment(
    arize_client=arize_client,
    dataset_id=DATASET_ID,
    name_prefix="v3-tool-agent",
    task=tasks["task_v3"],
    evaluators=EVALUATORS,
)
exp_v3 = run_v3["experiment"]
df_v3 = run_v3["results_df"]
print(f"  Done. Experiment ID: {exp_v3.id}")
print(f"  Rows evaluated: {len(df_v3)}")


---
## Step 5 — Production Readiness

Use the benchmark results in Arize to fill in this checklist.
The goal is not a perfect score. The goal is to know:
- where prompting alone is enough to improve the experience
- where static context is enough to answer well
- where only tools create the right customer experience
- which workflows are still missing the backend support needed for a real agent


In [ ]:
for row in format_checklist_rows(production_readiness_checklist()):
    print(row)
